# 04b Carbon: embodied carbon of the elements the scan confirms, and of the rest

**Slice.** The 35 room elements of the CRAS lab ("room B") with the verdicts from notebook 03a, and the materials of `craslabbim.ifc` (IFC2X3, metres).

**Question.** How many kg CO2e (product stage, A1-A3) are in the elements the scan confirms (OK, shifted), and how many in the rest, the elements the scan cannot vouch for?

Sections: (1) verdicts, material mapping and ÖKOBAUDAT datasets, (2) quantities from the IFC geometry, (3) carbon per element, (4) what is left out, (5) confirmed vs the rest, (6) sources, then the answer.
Terms (element, verdict, oracle labels) are defined in [CONTEXT.md](../CONTEXT.md).
Every number is an **A1-A3 estimate (product stage only)**: no transport to site, construction, use or end of life, and generic or industry-average datasets, not the products actually installed.
Factors come from [ÖKOBAUDAT](https://www.oekobaudat.de) (BBSR / BMWSB), fetched by `scripts/fetch_oekobaudat.py` into gitignored `data/`; only the material mapping `04b_material_mapping.csv` is committed, so factor values are not printed here, only quantities, results and the dataset citations.
Nothing is written outside `outputs/04b/` (gitignored), and the output is the same on every run.

In [1]:
import csv, os
from collections import Counter, defaultdict
from pathlib import Path

os.environ["OPEN3D_DISABLE_WEB_VISUALIZER"] = "true"  # before the import below: skips Open3D's Jupyter web-visualizer banner
import ifcopenshell
import ifcopenshell.util.element
import ifcopenshell.util.unit
import numpy as np

from scan_vs_bim.load import load_ifc_elements
from scan_vs_bim.oekobaudat import parse_dataset, reference_flow_uuid

# All parameters in one place.
IN = Path("../outputs/03a")  # written by notebook 03a
SRC_IFC = Path("../data/cras/craslabbim.ifc")
MAPPING = Path("04b_material_mapping.csv")  # committed: IFC material name -> ÖKOBAUDAT dataset UUID, no factor values
OEKO = Path("../data/oekobaudat")  # written by scripts/fetch_oekobaudat.py; gitignored
OUT = Path("../outputs/04b")  # gitignored; never write outside outputs/
OUT.mkdir(parents=True, exist_ok=True)
CONFIRMED = ("OK", "shifted")  # verdicts the scan vouches for; everything else is "the rest"
LABEL = "A1-A3 estimate (product stage only)"

## 1. Verdicts, material mapping and ÖKOBAUDAT datasets

The primary split uses the **no-labels** verdicts (what the pipeline gives on a real scan); the **oracle-labels** split is a second line and an upper bound, as in 04a.
The mapping gives each IFC material name a dataset UUID, or none. For every mapped UUID the process dataset and its reference-flow dataset are read from `data/oekobaudat/`: the flow holds how many kg or m3 one declared unit (m3, m2 or kg) contains, which is the only source of unit conversions here.

In [2]:
def read(variant):
    with open(IN / f"verdicts_{variant}.csv", newline="") as f:
        return {r["global_id"]: r for r in csv.DictReader(f)}


rows, oracle = read("no_labels"), read("oracle_labels")
assert len(rows) == 35 and set(rows) == set(oracle)
print("no labels:    ", dict(Counter(r["verdict"] for r in rows.values())))
print("oracle labels:", dict(Counter(r["verdict"] for r in oracle.values())))

with open(MAPPING, newline="", encoding="utf-8") as f:
    mapping = {r["ifc_material"]: r for r in csv.DictReader(f)}
assert all(not r["oekobaudat_uuid"] or r["basis"] in ("volume", "opening_area") for r in mapping.values())

datasets = {}
for uuid in sorted({r["oekobaudat_uuid"] for r in mapping.values() if r["oekobaudat_uuid"]}):
    process = OEKO / "processes" / f"{uuid}.xml"
    if not process.exists():
        raise FileNotFoundError(f"{process} is missing: run `uv run python scripts/fetch_oekobaudat.py` first")
    xml = process.read_text(encoding="utf-8")
    flow = OEKO / "flows" / f"{reference_flow_uuid(xml)}.xml"
    if not flow.exists():
        raise FileNotFoundError(f"{flow} is missing: run `uv run python scripts/fetch_oekobaudat.py` first")
    datasets[uuid] = parse_dataset(xml, flow.read_text(encoding="utf-8"))
    assert datasets[uuid].uuid == uuid
print(f"\n{len(mapping)} materials in the mapping, {sum(bool(r['oekobaudat_uuid']) for r in mapping.values())} of them mapped to {len(datasets)} ÖKOBAUDAT datasets")
print("declared units:", dict(Counter(d.declared_unit for d in datasets.values())))

no labels:     {'not confirmed': 29, 'shifted': 1, 'OK': 5}
oracle labels: {'not confirmed': 27, 'OK': 6, 'shifted': 2}

18 materials in the mapping, 15 of them mapped to 9 ÖKOBAUDAT datasets
declared units: {'m3': 5, 'm2': 3, 'kg': 1}


## 2. Quantities from the IFC geometry

The IFC has no quantity sets, so quantities come from the geometry:

- **Layered elements** (walls, slab, ceilings): the element's tessellated volume (signed tetrahedra of the closed mesh) times each layer's share of the total layer thickness. A dataset declared per m2 (the floor tile) gets the layer's area, its volume divided by its thickness.
- **Single-material elements** (one covering): the whole tessellated volume.
- **Windows and the door**: their meshes are not closed, so there is no per-material volume. The whole-product dataset is applied to the opening area, `OverallWidth x OverallHeight` from the IFC element, and all of one element's materials that map to the same dataset count **once**.

The whole element is counted, also where it extends beyond the room box (long walls, the slab, most ceilings): the verdicts are per element, so the carbon is too.

In [3]:
f = ifcopenshell.open(str(SRC_IFC))
assert f.schema == "IFC2X3" and ifcopenshell.util.unit.calculate_unit_scale(f) == 1.0, "layer thicknesses and Overall* must be in metres"
meshes = {m.global_id: m for m in load_ifc_elements(SRC_IFC, only_guids=list(rows))}
assert set(meshes) == set(rows)


def volume(mesh):
    """Signed volume of a closed triangle mesh, m3."""
    v = mesh.vertices[mesh.faces]
    return float(np.einsum("ij,ij->i", v[:, 0], np.cross(v[:, 1], v[:, 2])).sum() / 6)


# One item per material entry of an element: (material name, basis, available quantities in m3 / m2).
items = {}
for gid in rows:
    el, found = f.by_guid(gid), []
    material = ifcopenshell.util.element.get_material(el, should_skip_usage=True)
    if material.is_a("IfcMaterialLayerSet"):
        v = volume(meshes[gid])
        assert v > 0, f"{gid}: layered element does not tessellate to a closed solid"
        total = sum(layer.LayerThickness for layer in material.MaterialLayers)
        for layer in material.MaterialLayers:
            part = v * layer.LayerThickness / total
            found.append((layer.Material.Name, "volume", {"m3": part, "m2": part / layer.LayerThickness}))
    elif material.is_a("IfcMaterial"):
        v = volume(meshes[gid])
        assert v > 0, f"{gid}: element does not tessellate to a closed solid"
        found.append((material.Name, "volume", {"m3": v}))
    elif material.is_a("IfcMaterialList"):
        area = el.OverallWidth * el.OverallHeight
        found = [(m.Name, "opening_area", {"m2": area}) for m in material.Materials]
    else:
        raise ValueError(f"{gid}: unexpected material entity {material.is_a()}")
    items[gid] = found
missing = {name for found in items.values() for name, _, _ in found} - set(mapping)
assert not missing, f"materials not in the mapping: {sorted(missing)}"

by_class = defaultdict(list)
for gid, r in rows.items():
    by_class[r["ifc_class"]].append(gid)
for cls, gids in sorted(by_class.items()):
    if cls in ("IfcWindow", "IfcDoor"):
        areas = [items[g][0][2]["m2"] for g in gids]
        print(f"{cls:20s} {len(gids):2d} elements, opening area {sum(areas):6.2f} m2")
    else:
        vols = [volume(meshes[g]) for g in gids]
        print(f"{cls:20s} {len(gids):2d} elements, mesh volume {sum(vols):6.2f} m3")

IfcColumn             1 elements, mesh volume   0.15 m3
IfcCovering           5 elements, mesh volume   5.05 m3
IfcDoor               1 elements, opening area   3.30 m2
IfcSlab               1 elements, mesh volume  57.90 m3
IfcWallStandardCase  11 elements, mesh volume  29.13 m3
IfcWindow            16 elements, opening area  14.39 m2


## 3. Carbon per element

One row per material entry (`outputs/04b/carbon_layers.csv`), then per element (`outputs/04b/carbon_elements.csv`). A row ends in one of three states:

- **counted**: the dataset's declared unit, or a conversion given by its flow properties, turns the quantity into kg CO2e.
- **unmapped**: the mapping has no dataset for the material.
- **not quantifiable**: a dataset is mapped but the data holds no conversion from the available quantity (for example a dataset per kg and no density). Nothing is guessed.

In [4]:
layer_rows = []
for gid, found in items.items():
    seen = {}  # whole-product datasets count once per element
    for name, basis, q in found:
        m = mapping[name]
        uuid = m["oekobaudat_uuid"]
        if uuid and basis == "opening_area" and uuid in seen:
            seen[uuid]["ifc_material"] += f" + {name}"
            continue
        if not uuid:
            quantification, kg = "unmapped", None
        elif m["basis"] != basis:
            quantification, kg = "not quantifiable", None  # mapped for another basis than this element offers
        else:
            kg = datasets[uuid].carbon(q)
            quantification = "counted" if kg is not None else "not quantifiable"
        row = {"global_id": gid, "ifc_material": name, "oekobaudat_uuid": uuid, "basis": basis, "volume_m3": q.get("m3") if basis == "volume" else None,
               "area_m2": q.get("m2"), "kg_co2e_a1_a3": kg, "quantification": quantification}
        layer_rows.append(row)
        if uuid and basis == "opening_area":
            seen[uuid] = row


def num(x, fmt):
    return "" if x is None else format(x, fmt)


with open(OUT / "carbon_layers.csv", "w", newline="", encoding="utf-8") as fh:
    w = csv.writer(fh)
    w.writerow(["global_id", "ifc_material", "oekobaudat_uuid", "basis", "volume_m3", "area_m2", "kg_co2e_a1_a3", "quantification"])
    for r in layer_rows:
        w.writerow([r["global_id"], r["ifc_material"], r["oekobaudat_uuid"], r["basis"], num(r["volume_m3"], ".6f"), num(r["area_m2"], ".6f"),
                    num(r["kg_co2e_a1_a3"], ".3f"), r["quantification"]])

element_rows = []
for gid, r in rows.items():
    mine = [x for x in layer_rows if x["global_id"] == gid]
    left_out = sorted({x["ifc_material"] for x in mine if x["quantification"] != "counted"})
    element_rows.append({"global_id": gid, "ifc_class": r["ifc_class"], "name": r["name"], "verdict_no_labels": r["verdict"],
                         "verdict_oracle_labels": oracle[gid]["verdict"], "kg_co2e_a1_a3": sum(x["kg_co2e_a1_a3"] or 0 for x in mine),
                         "n_counted": sum(x["quantification"] == "counted" for x in mine), "left_out": "; ".join(left_out)})
with open(OUT / "carbon_elements.csv", "w", newline="", encoding="utf-8") as fh:
    w = csv.writer(fh)
    w.writerow(["global_id", "ifc_class", "name", "verdict_no_labels", "verdict_oracle_labels", "kg_co2e_a1_a3", "n_counted", "left_out"])
    for e in element_rows:
        w.writerow([e["global_id"], e["ifc_class"], e["name"], e["verdict_no_labels"], e["verdict_oracle_labels"], f"{e['kg_co2e_a1_a3']:.3f}",
                    e["n_counted"], e["left_out"]])

print(f"{LABEL}, kg CO2e per element (whole element, also beyond the room)\n")
print(f"{'class':20s} {'element':34s} {'no labels':14s} {'oracle labels':14s} {'kg CO2e':>9s}")
for e in sorted(element_rows, key=lambda e: (e["ifc_class"], -e["kg_co2e_a1_a3"])):
    short = e["name"] if len(e["name"]) <= 32 else e["name"][:24] + "~" + e["name"][-7:]
    print(f"{e['ifc_class']:20s} {short:34s} {e['verdict_no_labels']:14s} {e['verdict_oracle_labels']:14s} {e['kg_co2e_a1_a3']:9.1f}")
print(f"\n{len(layer_rows)} material entries: {dict(Counter(r['quantification'] for r in layer_rows))}")

A1-A3 estimate (product stage only), kg CO2e per element (whole element, also beyond the room)

class                element                            no labels      oracle labels    kg CO2e
IfcColumn            M_Concrete-Rectangular-C~1164179   not confirmed  not confirmed        0.0
IfcCovering          Compound Ceiling:Forro d~1428296   not confirmed  not confirmed      152.6
IfcCovering          Compound Ceiling:Forro d~1428995   not confirmed  not confirmed      136.2
IfcCovering          Compound Ceiling:Forro d~1431675   not confirmed  not confirmed      118.7
IfcCovering          Compound Ceiling:Forro d~1429616   not confirmed  not confirmed      117.0
IfcCovering          Forros 1:Forros 1:1537527          not confirmed  not confirmed        0.0
IfcDoor              Doors_Door-sets_Swedoor_~1044681   not confirmed  not confirmed      398.7
IfcSlab              Floor:(15cm) Piso + Porc~:911079   not confirmed  OK               24552.7
IfcWallStandardCase  Basic Wall:(35cm) T

## 4. What is left out

Materials the mapping leaves unmapped, and mapped materials the data cannot convert, with the elements they leave without that part of their carbon. Elements with no carbon at all (`counted` is 0) are listed separately: they appear as zero in the sums below, which is an under-count, not a measurement.

In [5]:
left = defaultdict(lambda: {"quantification": set(), "elements": []})
for r in layer_rows:
    if r["quantification"] != "counted":
        for name in r["ifc_material"].split(" + "):
            left[name]["quantification"].add(r["quantification"])
            left[name]["elements"].append(r["global_id"])
for name, d in sorted(left.items()):
    classes = dict(Counter(rows[g]["ifc_class"] for g in d["elements"]))
    print(f"{name:42s} {'/'.join(sorted(d['quantification'])):17s} {len(d['elements']):2d} entries in {classes}")
    print(f"{'':42s} {mapping[name]['note']}")
none = [e for e in element_rows if e["n_counted"] == 0]
print(f"\nelements with no carbon counted at all: {len(none)}")
for e in none:
    print(f"  {e['ifc_class']:20s} {e['name'][:50]:52s} {e['verdict_no_labels']}")
used = {name for found in items.values() for name, _, _ in found}
unused = sorted(n for n, r in mapping.items() if r["oekobaudat_uuid"] and n not in used)
print("\nmapped materials that no room element uses:", unused or "none")

<Unnamed>                                  unmapped           6 entries in {'IfcColumn': 1, 'IfcWindow': 5}
                                           No material name (the column, and one entry in each Westcoast window material list): nothing to map
ExpandingFoam_ALUPROF_Yellow               unmapped           1 entries in {'IfcWindow': 1}
                                           Mounting foam on one window: not a product the whole-window dataset describes; left out
Pintura branca                             not quantifiable   3 entries in {'IfcCovering': 1, 'IfcWallStandardCase': 2}
                                           Declared per kg and the data gives no density: expected to come out as not quantifiable
Rubber - Westcoast Windows - Light Grey    unmapped           5 entries in {'IfcWindow': 5}
                                           Window gasket: part of the whole window already counted through the aluminium frame

elements with no carbon counted at all: 2
  IfcColumn  

## 5. Confirmed vs the rest

Confirmed is OK + shifted: the scan shows the element is there (shifted ones are built, only displaced). The rest (missing, not visible, not confirmed) is carbon the model claims but the scan cannot vouch for. The **difference** is that second number: it is the part of the model's carbon that stands on the model alone. With only 03a's verdicts, "not confirmed" mostly means too few scan points, not that the element is absent. Both splits are written to `outputs/04b/carbon_summary.csv`.

In [6]:
kg_of = {e["global_id"]: e["kg_co2e_a1_a3"] for e in element_rows}
total = sum(kg_of.values())


def summary(title, verdicts):
    conf = [g for g in rows if verdicts[g]["verdict"] in CONFIRMED]
    rest = [g for g in rows if verdicts[g]["verdict"] not in CONFIRMED]
    kg_conf, kg_rest = sum(kg_of[g] for g in conf), sum(kg_of[g] for g in rest)
    assert abs(kg_conf + kg_rest - total) < 1e-6 and len(conf) + len(rest) == len(rows)
    print(f"{title}: {LABEL}")
    print(f"  {'':34s} {'elements':>8s} {'kg CO2e':>10s} {'share':>7s}")
    print(f"  {'confirmed (OK + shifted)':34s} {len(conf):8d} {kg_conf:10.0f} {kg_conf / total:7.1%}")
    print(f"  {'the rest':34s} {len(rest):8d} {kg_rest:10.0f} {kg_rest / total:7.1%}   {dict(Counter(verdicts[g]['verdict'] for g in rest))}")
    print(f"  {'total (all 35 room elements)':34s} {len(rows):8d} {total:10.0f}")
    print(f"  difference, the rest the scan cannot vouch for: {kg_rest:.0f} kg CO2e\n")
    return kg_conf, kg_rest


print("Confirmed by class, no labels:", {c: round(sum(kg_of[g] for g in gs if rows[g]["verdict"] in CONFIRMED)) for c, gs in sorted(by_class.items())}, "\n")
conf_no, rest_no = summary("No labels (primary)", rows)
conf_or, rest_or = summary("Oracle labels (upper bound)", oracle)

with open(OUT / "carbon_summary.csv", "w", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["variant", "confirmed_elements", "confirmed_kg_co2e_a1_a3", "rest_elements", "rest_kg_co2e_a1_a3", "total_kg_co2e_a1_a3"])
    for variant, verdicts, kg_conf, kg_rest in (("no_labels", rows, conf_no, rest_no), ("oracle_labels", oracle, conf_or, rest_or)):
        n_conf = sum(verdicts[g]["verdict"] in CONFIRMED for g in rows)
        w.writerow([variant, n_conf, f"{kg_conf:.1f}", len(rows) - n_conf, f"{kg_rest:.1f}", f"{total:.1f}"])

Confirmed by class, no labels: {'IfcColumn': 0, 'IfcCovering': 0, 'IfcDoor': 0, 'IfcSlab': 0, 'IfcWallStandardCase': 105, 'IfcWindow': 511} 

No labels (primary): A1-A3 estimate (product stage only)
                                     elements    kg CO2e   share
  confirmed (OK + shifted)                  6        616    1.6%
  the rest                                 29      38017   98.4%   {'not confirmed': 29}
  total (all 35 room elements)             35      38632
  difference, the rest the scan cannot vouch for: 38017 kg CO2e

Oracle labels (upper bound): A1-A3 estimate (product stage only)
                                     elements    kg CO2e   share
  confirmed (OK + shifted)                  8      25275   65.4%
  the rest                                 27      13357   34.6%   {'not confirmed': 27}
  total (all 35 room elements)             35      38632
  difference, the rest the scan cannot vouch for: 13357 kg CO2e



## 6. Sources

Each factor used, cited by dataset. Source: **ÖKOBAUDAT**, the German federal building-LCA database (BBSR / BMWSB), https://www.oekobaudat.de; datasets used unmodified, indicator GWP-total (kg CO2e), modules A1-A3.

In [7]:
use = Counter(r["oekobaudat_uuid"] for r in layer_rows if r["oekobaudat_uuid"])
print(f"{LABEL}; source: ÖKOBAUDAT (https://www.oekobaudat.de)\n")
for uuid, n in sorted(use.items(), key=lambda kv: datasets[kv[0]].name):
    d = datasets[uuid]
    print(f"{d.name}\n    UUID {uuid}, version {d.version}, declared unit {d.declared_unit}, used in {n} material entries")

A1-A3 estimate (product stage only); source: ÖKOBAUDAT (https://www.oekobaudat.de)

Applicationpaint emulsion, dispersion paint
    UUID 8c5e949d-1b5e-4dc6-b6ae-bfc8e6c863ce, version 20.26.070, declared unit kg, used in 3 material entries
Cement mortar
    UUID 96171066-34d9-4b33-b1e6-3af1ea11db61, version 20.26.070, declared unit m3, used in 2 material entries
Exterior door - VFF - Steel exterior door with light cut-out
    UUID 54a4dad1-bc25-4384-acf4-87ed9bab0026, version 00.00.033, declared unit m2, used in 1 material entries
Facing brick
    UUID 1401638f-b9d7-4b33-9444-b072e2756f49, version 20.26.070, declared unit m3, used in 1 material entries
Gypsum interior plaster
    UUID 7e0c008a-009d-4357-92c1-afb2cec3bc0a, version 20.26.070, declared unit m3, used in 24 material entries
Lime-cement plaster
    UUID dca9f05c-036b-40e5-ae81-c957c2420fd8, version 20.26.070, declared unit m3, used in 2 material entries
Mineral wool (partition walls insulation)
    UUID a5b22bbe-175d-4aef-bb1

## Answer

The scan confirms (OK or shifted) 6 of the 35 room elements without oracle labels, and they hold 616 kg CO2e of the model's 38632 kg CO2e (A1-A3 estimate, product stage only). The other 29 elements, all "not confirmed", hold 38017 kg CO2e: that is the difference, the carbon that rests on the model alone. With oracle labels 8 elements are confirmed with 25275 kg CO2e and 27 are not, with 13357 kg CO2e; the oracle split is an upper bound. The no-labels split is the reported one.

Both splits are dominated by a few large elements, not by the count of confirmed ones. The floor slab alone is 24552.7 kg CO2e and the 35 cm masonry wall 9935.3 kg CO2e (whole elements, also beyond the room). The slab is not confirmed without labels and OK with them, which is why the two splits differ so much; the masonry wall is not confirmed in either. So the split follows the verdicts of the slab and the masonry wall, and the 29 "not confirmed" elements are not evidence that anything is missing.

How far to trust the totals: of 72 material entries, 57 were counted, 12 are unmapped and 3 are not quantifiable (section 4). The column (unnamed material) and one ceiling covering (only paint, which ÖKOBAUDAT declares per kg with no density) get no carbon at all, so they count as zero, which is an under-count. Two stand-ins are judgement calls recorded in `04b_material_mapping.csv`: the masonry wall's brick layer uses a solid-brick dataset for a hollow ceramic block, which probably overstates it, and the slab's screed layer uses a mortar dataset, whose effect is not known. Quantities come from the tessellated IFC geometry (windows and the door from the opening area), and every dataset is cited in section 6 with its UUID and version; factor values are not committed or printed.